# 🔍 Block 4 — Can You Trust This Model?
## Inspecting data & results *before* believing an ML system

> **The story.** You're the **plant manager at a tool-manufacturing company.** Out on the
> factory floor, machines cut and shape metal parts all day — and when one breaks down
> mid-production, it's expensive. So your engineering team built a model that tries to predict
> a machine failure *before* it happens. They just demoed it with a headline number:
> **"99%+ accuracy — ready to ship Monday!"**
> Before you sign off on deploying it across the plant, you decide to **open the hood**.

This notebook walks you through what a careful practitioner checks before trusting *any*
ML result: the **data**, the **labels**, and the **metrics**. Along the way you'll learn the
**pandas** moves — `head`, `info`, `describe`, `value_counts`, boolean filtering — that data
scientists reach for every day. By the end you'll know exactly why the "99%" was a mirage.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- Every task is followed by a quick **self-check** so you know it worked.
- Whenever you need a specific function, we tell you which one. 🙂
- A few **no-code interactive widgets** let you explore by clicking.


## 0. Setup

This notebook is **self-contained**: when you run the first cell on Colab it clones the rest of
the exercise files (the `pdm_viz.py` display helpers and the factory dataset under `data/`)
directly from the public course repository. Run the setup cells below in order.

> ✅ The course repo is **public**, so no GitHub token is needed — the cell clones it directly.
> If you're running this notebook from inside a local clone of the repo, the setup cell skips the
> clone and just uses the files already on disk.

**0.1 — Fetch the exercise files.**

In [1]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE0-public"      # public repo — no GitHub token required

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding both the `exercises/` helpers and the
# `data/` files — so the data later loads with a clean `pd.read_csv("data/…")`.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()), os.getcwd()]:
    if (os.path.exists(os.path.join(_root, "exercises", "pdm_viz.py"))
            and os.path.isdir(os.path.join(_root, "data"))):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (exercises/ + data/). On Colab, re-run this cell to clone it; "
        "locally, run the notebook from inside a clone of the FDD-WE0-public repo.")
sys.path.insert(0, os.path.join(os.getcwd(), "exercises"))   # make the helpers importable
print("Working directory:", os.getcwd())

Cloning the exercise repo…
Working directory: /content/FDD-WE0-public


**0.2 — Install dependencies.** All of these are already on Colab; this just guarantees
the right versions (and makes the notebook work outside Colab too).

In [2]:
%pip install -q -r exercises/requirements_block4.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.6 MB/s eta 0:00:00


**0.3 — Import the libraries** we'll use throughout. The visual widgets and a few display
settings live in **`pdm_viz`** so these cells stay about *pandas*, not about HTML.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

import json
import importlib
import pdm_viz            # display settings, the heatmap helper, and all the click-widgets
importlib.reload(pdm_viz)    # pick up the latest helpers even if a stale copy was cached
pdm_viz.configure_display()

# The five physical sensor measurements — we'll reuse this list throughout.
NUMERIC = pdm_viz.NUMERIC

print("Environment ready ✅")

Display configured ✅
Environment ready ✅


In [6]:
pdm_viz.pipeline_diagram()

### The engineering team's result

They didn't hand us much — just a slide. Here is the **whole pipeline** they ran, and the
number they put on it: they threw **every usable column** at the model, patched the gaps,
trained, and reported **≈ 100% accuracy**.

In [7]:
pdm_viz.engineers_pipeline()

Rather than take their code at face value, we'll **open up each block in turn** — load,
fill, encode, train, score — and check whether it holds. (For instance: when we reach *"load
the CSV"*, we'll find they took the file exactly as-is — which is where the trouble starts.)

In [8]:
pdm_viz.bait_banner()

---
# Part 1 — Inspect the DataFrame

Before any modeling, understand *what you are looking at*.

- A **DataFrame** is a table: **rows = observations**, **columns = variables**.
- Here, **one row = one snapshot of a machine during a production cycle** (its sensor
  readings at that moment, plus whether it failed).
- The model's claim is simple: **given a machine's sensor readings (the features `X`),
  predict whether it will fail (the target `y`)**. It learns this mapping **`X → y`** from the
  historical rows, betting that the pattern it finds will still hold on machines it has never
  seen. So our first job is just to *get to know the table*: what each column is, and which ones
  are even usable.

### Load it — one line of pandas

The factory data ships as two CSV files: **`historical`**, the batch of past production records
the team built the model on, and **`deployment`**, a second batch we'll set aside and only come
back to near the very end. For everything that follows we work with `historical`.

Reading a CSV into a DataFrame is a one-liner you'll use constantly — **`pd.read_csv(path)`**.

> 💡 **Why "messy"?** The `messy_` in the filenames isn't a hint that anything was rigged — it
> just means this is **raw, unprocessed** data, exported straight from the factory exactly as it
> was recorded, with no cleaning applied yet. Turning that raw data into something you can trust
> is the whole point of this notebook.

In [9]:
historical_df = pd.read_csv("data/pdm_messy_historical.csv")
deployment_df = pd.read_csv("data/pdm_messy_deployment.csv")

print("historical:", historical_df.shape, "· deployment:", deployment_df.shape)

historical: (8150, 11) · deployment: (2000, 11)


### First moves on any DataFrame

Two things you'll reach for immediately:
- **`df.columns`** — the list of column names.
- **`df["col name"]`** — pull out a single column → you get a **Series**; if you pass a *list* of
  column names, you get a smaller **DataFrame** back.

In [10]:
print("Columns:", list(historical_df.columns))

# one column -> a Series:
print("\nThe Torque column (first 5 values):")
print(historical_df["Torque [Nm]"].head())

Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'Failure Type', 'Inspectors']

The Torque column (first 5 values):
0    44.7
1    24.4
2    35.8
3    46.5
4    31.2
Name: Torque [Nm], dtype: float64


### 🎯 Quick check — pull out a column yourself
Same move as above, on two other columns. Grab the **`Air temperature [K]`** and the
**`Tool wear [min]`** columns (first 5 values each).

> 💡 **`.head()`** shows 5 rows by default, but you can pass a number to choose how many —
> e.g. `.head(10)` for the first ten. Try a different number if you're curious.

In [13]:
# 🎯 the Air temperature column (first 5 values)
print("Air temperature:")
print("\nAir temperature [K] (first 5 values):")        # 🎯 hint: what is the exact name of the Air temperature column? (then .head())

# 🎯 the Tool wear column (first 5 values)
print("\nTool wear:")
print(historical_df["Tool wear [min]"].head(10))        # 🎯 hint: same move — what is the exact name of the Tool wear column?

Air temperature:

Air temperature [K] (first 5 values):

Tool wear:
0    179.0
1    113.0
2    196.0
3     25.0
4    199.0
5     28.0
6    165.0
7     92.0
8    140.0
9     22.0
Name: Tool wear [min], dtype: float64


Selecting columns keeps a **subset of the columns, every row**:

In [14]:
pdm_viz.column_select_demo()

Type,Air temp,Torque,failure
L,298.1,42.8,0
M,998.6,46.3,0
H,300.4,-9.1,1
L,299.0,NaN,0


And it's worth knowing **which way a pandas method runs**: some *aggregate down a column*
(many values → one number, like `min`/`max`/`mean`), others *transform each cell* and hand back
the same shape (like `isna`). Keep this picture in mind — we'll use both.

In [15]:
pdm_viz.axis_demo()

**The two directions chain beautifully.** `isna()` *transforms* each cell into `True`/`False`
(`True` exactly where a value is missing); `True` counts as **1**, so following it with `.sum()`
*aggregates* those booleans into a count. This **`.isna().sum()`** combo — count the missing values
— is one you'll reach for constantly, so let's watch it on a tiny Series now.

In [16]:
toy = pd.Series([10, None, 7, None])
print(toy.isna())                      # True exactly where a value is missing
print("missing values:", toy.isna().sum())   # True counts as 1 -> a count

0    False
1     True
2    False
3     True
dtype: bool
missing values: 2


**`.head()`** shows the first few rows — your first look at the raw data.

In [17]:
historical_df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Failure Type,Inspectors
0,3176,L50355,L,300.3,309.6,1468,44.7,179.0,0,No Failure,"{""lead"": ""Hofmann"", ""team"": [""Hofmann""], ""shif..."
1,4514,L51693,L,302.4,310.3,1900,24.4,113.0,0,No Failure,"{""lead"": ""Alvarez"", ""team"": [""Alvarez"", ""Chen""..."
2,1672,L48851,L,298.1,307.9,1614,35.8,196.0,0,No Failure,"{""lead"": ""Alvarez"", ""team"": [""Alvarez"", ""Fisch..."
3,343,L47522,L,297.4,308.2,1471,46.5,25.0,0,No Failure,"{""lead"": ""Jensen"", ""team"": [""Jensen""], ""shift""..."
4,2078,L49257,L,299.4,309.4,1701,31.2,199.0,0,No Failure,"{""lead"": ""Hofmann"", ""team"": [""Hofmann""], ""shif..."


### 🕵️ One column looks odd

Most columns are single numbers or short labels. But **`Inspectors`** is different — peek at a
single cell to see what's inside.

In [18]:
one_cell = historical_df["Inspectors"].iloc[0]
print("value:", one_cell)
print("type :", type(one_cell))   # 👀 a whole record packed into one cell, as JSON text
# Note that this kind of nesting is very common in practice.

value: {"lead": "Hofmann", "team": ["Hofmann"], "shift": "day", "date": "2024-12-22", "all_passed": false}
type : <class 'str'>


So one cell holds a whole **record** — lead inspector, the team, the shift, a date, a
pass/fail flag — packed into a single string of **JSON**. To work with it, we **unnest** it:
`json.loads` turns each JSON string into a Python `dict`, and **`pd.json_normalize(...)`** spreads
a column of those dicts into a tidy table — one key per column. We'll run that for you:

In [19]:
records = historical_df["Inspectors"].apply(json.loads)   # JSON text -> dicts
inspectors_df = pd.json_normalize(records)                # one column per key
display(inspectors_df.head())

,lead,team,shift,date,all_passed
0,Hofmann,[Hofmann],day,2024-12-22,False
1,Alvarez,"[Alvarez, Chen, Iqbal]",night,2024-03-17,False
2,Alvarez,"[Alvarez, Fischer, Jensen]",swing,2024-11-29,True
3,Jensen,[Jensen],day,2024-07-30,True
4,Hofmann,[Hofmann],day,2024-05-01,True


### 🎯 Your turn — explore it, then judge it

You already know how to list a table's columns — **`df.columns`**. Use it on `inspectors_df` to
see exactly which fields this packed column was hiding. Then look at those names and ask the
question that really matters: **would any of this help a model predict a machine failure?**

In [21]:
print(list(inspectors_df.columns))

['lead', 'team', 'shift', 'date', 'all_passed']


In [ ]:
pdm_viz.mc_quiz("inspectors_useful")

Right — it's **inspection bookkeeping** (who signed off, when), no sensor signal about
whether the machine fails. We'll **drop** it from the modeling table shortly.

### 📖 What does each column even mean?

Click the columns you don't recognise to read what they are.

In [ ]:
pdm_viz.data_dictionary(historical_df.columns)

**`.info()`** summarises the columns: their names, how many non-null values each has, and
their data type (`dtype`). Already a quick way to spot columns with missing data.

In [ ]:
print("Shape:", historical_df.shape)
historical_df.info()

### Task 1 — explore the categories

Numbers you can average; categories you **count**. `Failure Type` is categorical, so three
pandas moves are your bread and butter for it:
- **`.unique()`** — the list of *distinct* values (no counts).
- **`.count()`** — how many *non-null* values there are in total (one number).
- **`.value_counts()`** — how many times *each* distinct non-null value appears.

Start with the first two — just run this:

In [ ]:
print("Distinct failure types:", historical_df["Failure Type"].unique())
print("Non-null values total:", historical_df["Failure Type"].count())

### 🔮 Before you trust these, predict them

`.unique()`, `.count()`, `.nunique()` and `.value_counts()` look similar but answer **different**
questions — and they don't all treat a missing value the same way. (And no, `.nunique()` is not a typo: it returns the **number** of distinct values — a single count — while `.unique()` returns the distinct values *themselves*.) In the widget below, **type the
number** each counting call returns (it'll tell you green/red), then reveal the list-valued ones.

In [ ]:
pdm_viz.predict_unique_count()

### The per-value breakdown — `.value_counts()`

`.value_counts()` lists how often *each* distinct value appears. Run it on `Failure Type`:

In [ ]:
print(historical_df["Failure Type"].value_counts())

Notice how lopsided that is — **`No Failure`** dwarfs everything else. Keep that in mind.

### 🎯 Task 2 — drop the nested column

We've seen what's inside `Inspectors`: bookkeeping, no measurement signal. Leaving a packed
JSON column in the modeling table only invites trouble (a model could even memorize inspector
names). Drop it with **`.drop(columns=[...])`** and keep the result in `historical_df`.

In [ ]:
# 🎯 drop the nested 'Inspectors' column
historical_df = historical_df.drop(columns=[ ??? ])   # 🎯 the nested column

assert "Inspectors" not in historical_df.columns
print("Columns now:", list(historical_df.columns))

### `.describe()` — the all-in-one summary

Now that the nested column is gone, **`.describe(include="all")`** gives count, mean, std, min/max
and quartiles for numeric columns, plus unique/top/frequency for categorical ones. Crucially, the
result is **just another DataFrame** — so you can slice it like any other. **`.T`** *transposes* it
(flips rows ↔ columns); we add it so each original column becomes a row, which reads more easily
here. Because it's a DataFrame, e.g. `historical_df.describe(include="all")["Type"]` pulls out the
summary of a single column.

> 💡 **Why all the `NaN`s?** `describe` shows *every* statistic in every row. A numeric column
> has no "most frequent category", and a text column has no "mean" — so the cells that don't
> apply are simply left blank (`NaN`). It's not missing *data*, just a non-applicable *statistic*.

In [ ]:
historical_df.describe(include="all").T

### 🧠 Quick check — what does the summary tell you?

In [ ]:
pdm_viz.true_false_quiz("describe")

### 🎯 Task 3 — name the target

Our job is to predict **whether a machine fails** — a plain **yes/no** outcome. Here are all the
columns again; thinking about *that* question, decide **which single column is the clean binary
label** we should try to predict.

> 💡 Some columns are inputs (sensor readings), some are identifiers, and a couple touch on
> failure in different ways. You're after the one that is exactly a 0/1 "did it fail?" flag —
> not a description of *how* it failed. Set `TARGET` to that column's name.

In [ ]:
print("Columns to choose from:")
for c in historical_df.columns:
    print("  -", c)

In [ ]:
# 🎯 Which column is the binary thing we are trying to predict?
TARGET = ???          # 🎯 fill in the column name (a string)

# --- self-check ---
assert TARGET in historical_df.columns, "That column isn't in the DataFrame."
assert historical_df[TARGET].nunique() == 2, "The target should be binary (0/1)."
print("Target column:", TARGET, "· values:", sorted(historical_df[TARGET].unique()))
print("Failure rate:", round(historical_df[TARGET].mean(), 4))

### 🎯 Task 4 — slice the table to investigate

The real power of pandas is **boolean filtering**: `df[condition]` keeps only the rows where
`condition` is `True`. It's the **mirror image** of selecting columns — this time we keep a
subset of the *rows*, and *every* column:

In [ ]:
pdm_viz.row_filter_demo()

Let's use it to look at the rows with a **missing Torque** reading — pass
**`historical_df["Torque [Nm]"].isna()`** as the condition.

In [ ]:
# 🎯 keep only the rows where Torque is missing (NaN)
torque_missing = historical_df[ ??? ]      # 🎯 which method flags the missing cells? (you met it in the axis demo)

print("Rows with a missing Torque reading:", len(torque_missing))
torque_missing.head()

### 🧩 Challenge — build your own `value_counts()`

Remember `value_counts()` from earlier? It isn't magic: it just walks the **distinct** values and,
for each one, **counts the rows that match** — which is exactly the boolean-mask move you just used
above. Rebuild it yourself with a `for` loop over `.unique()`, filling a dictionary as you go.

> 💡 This works cleanly here because `Failure Type` has **no missing values**, so `.unique()` and
> `.value_counts()` see the same set of labels. On a column *with* NaNs they'd disagree (recall:
> `.unique()` keeps NaN, `.value_counts()` drops it) — you'd have to skip the NaN to match.

In [ ]:
my_counts = {}
for value in # 🎯 which values are we iterating over?
    # hint1: first keep the rows whose Failure Type equals `value` (the boolean filtering above)
    # hint2: in a .sum(), True counts as 1 and False as 0 — so summing a mask counts the matches
    my_counts[value] = ???

print(my_counts)
# does it match the real thing?
assert my_counts == historical_df["Failure Type"].value_counts().to_dict()
print("\nMatches value_counts() ✅")

### 🕹️ Exploration challenge

Each answer below is **one slice away**. Write your own little queries in the scratch cell, then
type the numbers in. (Reminder: `(df[col] > x).sum()` counts how many rows satisfy a condition.)

In [ ]:
# 🧪 scratch cell — build your own queries here.
# Reminder of the pattern: a comparison makes a True/False mask, and .sum() counts the Trues.
# Worked example on a column the quiz does NOT ask about — adapt it to each question:
(historical_df["Process temperature [K]"] > 310).sum()

In [ ]:
pdm_viz.number_quiz("exploration")

### 🕹️ Interactive — what is each column's *role*?

You now know the columns. Not every one is a usable feature. Click a column, then click the role
you think it plays:

- **feature** — a legitimate input available *before* we know the outcome.
- **target** — the thing we predict.
- **ID / artifact** — an arbitrary identifier (row number, product key). No predictive meaning.
- **leakage** — information that *encodes the answer* or wouldn't exist at prediction time.

Get all 10 right. 👇

In [ ]:
pdm_viz.column_role_game()

**Two columns deserve a hard look:**

- **`UDI`** and **`Product ID`** are just identifiers. A model could memorize them, but they
  carry no real-world signal — classic **artificial features**.
- **`Failure Type`** describes *how* the machine failed (e.g. *Heat Dissipation Failure*).
  That information only exists **after** a failure has happened, and `No Failure` marks every
  healthy row — so it *encodes the answer*. Using it to predict *whether* a failure happens is
  textbook **target leakage**, and it's exactly what handed the engineers their fake ~100%.

### 🧠 Wrap-up quiz — pick the right pandas move

You've met the core pandas operations. For each situation, click the operation that does the job.
Watch out — some options are *close but wrong* (a mask isn't a count; selecting a column isn't
filtering rows).

In [ ]:
pdm_viz.operations_quiz()

---
# Part 2 — Hunt for data problems

The engineers trained on this file as-is. Let's check it the way you'd check any dataset:
**missing values, duplicates, inconsistent categories, impossible values,** and **leakage**.

### 🎯 Task 5 — missing values

Time to put the **`.isna().sum()`** combo from the start of Part 1 to work — this time across the
**whole DataFrame**. `historical_df.isna()` gives a `True`/`False` for every cell, and `.sum()`
then adds them up **per column** — one missing-count for each column.

In [ ]:
# 🎯 count missing (NaN) values in each column
missing_per_column = ???     # 🎯 hint: you'll need both .isna() and .sum()

print(missing_per_column)
assert missing_per_column.sum() > 0, "There ARE missing values — check your method."
print("\nColumns with gaps:", missing_per_column[missing_per_column > 0].index.tolist())

### 🎯 Task 6 — duplicate rows (and *why* they're poison)

A new tool for this one: **`.duplicated()`** scans the rows top to bottom and returns `True` for any
row that is a **repeat of one already seen above** (the first time a value appears it's `False`).
That's the same shape-preserving idea as `.isna()` — a `True`/`False` per row — so, exactly like
before, chaining **`.sum()`** counts the repeats. See it on a tiny list of IDs first:

In [ ]:
toy_ids = pd.Series([1, 2, 2, 3, 1])
print(toy_ids.duplicated())                      # True where a value was already seen above
print("repeats:", toy_ids.duplicated().sum())    # count them with .sum(), just like missing values

Now the real check. `UDI` is supposed to be a **unique key** — one row per device inspection.
So if the *same* `UDI` shows up twice, those are copy-paste duplicates. Count rows that repeat an
already-seen `UDI` by passing **`subset="UDI"`** to `.duplicated()`, and compare it to the count of
fully-identical rows (that's `.duplicated()` with no argument — every column must match).

In [ ]:
n_exact = historical_df.duplicated().sum()             # fully identical rows

# 🎯 how many rows repeat a UDI we've already seen?
n_dup_udi = historical_df.duplicated(subset=???).sum()  # 🎯 the unique-key column name

print("Exact duplicate rows:", n_exact)
print("Rows repeating a UDI :", n_dup_udi)
assert n_dup_udi == n_exact, "If these match, the duplicates really are whole-row copies."

# 👀 the 'aha': look at one offending UDI — same device, same readings, twice.
dup_udi = historical_df.loc[historical_df.duplicated(subset="UDI", keep=False), "UDI"].iloc[0]
display(historical_df[historical_df["UDI"] == dup_udi])

**Why this matters:** that exact row could land in *both* the training and the test split.
The model would then be "tested" on a row it already memorized — inflating its score for free.
We'll drop these **before** splitting the data.

### 🎯 Task 7 — inconsistent categories

`Type` should be one of three machine grades: **L** (low), **M** (medium), **H** (high). To see
whether that's really all that's in the column, you want the count of **how often each distinct
value shows up** — the same per-value tally you reached for back in Part 1.

In [ ]:
# 🎯 show how often each distinct value of `Type` appears
print(???)              # 🎯 which single method tabulates each distinct value with its count?

See the problem? The same three grades are written **many different ways** — `L`, `l`, `Low`
all mean the same machine. To a model these look like *different* categories. We'll fix this
with a **canonical mapping** (a dictionary that maps every spelling to one clean label):

In [ ]:
TYPE_CANON = {"L": "L", "l": "L", "Low": "L",
              "M": "M", "Medium": "M",
              "H": "H", "High": "H"}

cleaned_type = historical_df["Type"].map(TYPE_CANON)
print("Before:", sorted(map(str, historical_df['Type'].unique())))
print("After :", sorted(cleaned_type.unique()))
assert cleaned_type.isna().sum() == 0, "Some raw value wasn't covered by the mapping!"
print("\nClean distribution:\n", cleaned_type.value_counts().to_string())

### Combining conditions — `&` (and) and `|` (or)

So far each filter used **one** condition. Real checks often need several at once, and two
True/False masks combine **element-wise**:
- **`&`** (and) → `True` only where **both** masks are `True`.
- **`|`** (or) → `True` where **at least one** mask is `True`.

One catch: wrap **each** comparison in **parentheses**, because `&`/`|` bind tighter than `<`, `>`,
`==`. Watch it on a tiny Series:

In [ ]:
a = pd.Series([1, 5, 9])
print((a > 2) & (a < 8))    # AND: 2 < value < 8   -> only the middle one is True
print((a < 2) | (a > 8))    # OR : very small OR very large -> the two ends are True

### 🎯 Task 8 — impossible values

Sensors fail too. Some rows hold **physically impossible** readings. Build a boolean mask that
flags rows where **any** of these is true:
- `Torque [Nm]` is **negative**, or
- `Rotational speed [rpm]` is **exactly 0** (a running machine can't be at 0 rpm), or
- `Air temperature [K]` is **above 500** (≈ 227 °C — a stuck sensor).

In [ ]:
# 🎯 mask = True for rows with at least one impossible reading
impossible = (
    (historical_df["Torque [Nm]"] < 0)
    ???      # 🎯 add the Rotational speed and Air temperature conditions — which operator means "any can be true"?
)

print("Impossible rows found:", int(impossible.sum()))
assert impossible.sum() >= 8
display(historical_df.loc[impossible, NUMERIC])

### 🎯 ...and how many rows have *all three* problems at once?

The mask above flagged a row if **any** single fault was present. Now demand **all three at the
same time**. Think about what you expect *before* you run it.

In [ ]:
# 🎯 mask = True only for rows that hit ALL THREE problems simultaneously
all_three = (
    (historical_df["Torque [Nm]"] < 0)
    ???      # 🎯 require the other two conditions too — which operator means "all must be true"?
)

print("Rows with all three problems at once:", int(all_three.sum()))

**Zero.** No single row trips all three at once — each broken sensor shows up on its *own*
row. That's the point of `&` vs `|`: `&` is far stricter, demanding every condition together, so it
matches far fewer rows (here, none). For *flagging* bad data we want `|` — catch a row if **any**
reading is impossible.

### 🎯 Task 9 — fill in the data quality report card

You've now measured every problem. Plug your numbers into the report card. Each value is a small
computation you've already seen — fill the blanks and run it.

In [ ]:
# 🎯 fill each value from what you computed above
report_missing    = ???    # 🎯 total missing cells in the WHOLE frame (isna + sum — but one sum gives a per-column count; sum *that* again for the grand total)
report_duplicates = ???    # 🎯 exact duplicate rows (the .duplicated() count from Task 6)
report_spellings  = int(historical_df["Type"].nunique() - 3)   # extra `Type` spellings beyond L/M/H — given for you
report_impossible = ???    # 🎯 impossible sensor rows — you already stored that mask above

pdm_viz.quality_report_card(report_missing, report_duplicates,
                            report_spellings, report_impossible)

---
# Part 3 — Visualize the data

Numbers in a table hide things that a picture reveals instantly.

**Histograms of the sensor features.** A histogram buckets values into bars so you can see
the *shape* of a column. Watch for skew and stray spikes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.ravel(), NUMERIC):
    ax.hist(historical_df[col].dropna(), bins=40, color="#6f7bf0", edgecolor="white")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("measured value"); ax.set_ylabel("number of rows")
axes.ravel()[-1].axis("off")
fig.suptitle("Distributions of the sensor features", fontweight="bold")
plt.tight_layout(); plt.show()

### 🧠 Quick check — what do the histograms tell you?

One plot looks suspicious: **Air temperature** is a single tall bar. Did the machines really all
run at one temperature? Think about what you found in Task 8 before you answer.

In [ ]:
pdm_viz.true_false_quiz("histograms")

### 🎯 Your turn — strip the outliers and re-plot Air temperature

You diagnosed it: the lonely spike is just the **~999 K stuck-sensor** readings stretching the
axis. Prove it — filter them out (you already know how: a boolean mask), then re-draw the
histogram on the **real** range. Keep only rows where `Air temperature [K]` is **below 500**.

In [ ]:
# 🎯 keep the physically-plausible air-temperature readings (below 500 K).
# Same two moves you already know: filter the rows with a boolean mask, then pick the column.
air_rows = historical_df[ ??? ]                    # 🎯 which mask corresponds to the predicate above? (keep rows below 500 K)
air_ok = air_rows["Air temperature [K]"]

print("Dropped", historical_df["Air temperature [K]"].notna().sum() - air_ok.notna().sum(),
      "stuck-sensor rows")
plt.hist(air_ok.dropna(), bins=40, color="#6f7bf0", edgecolor="white")
plt.title("Air temperature — outliers removed"); plt.xlabel("K"); plt.show()

With the stuck-sensor rows gone, the *real* spread of air temperature finally shows — a
sensible bell-ish shape around ~300 K, not one giant bar. Same data, honest picture.

### A new plot to add to your kit — `plt.bar`

The histograms above were drawn for you. Here's the one new plotting call you'll write yourself: a
**bar chart** takes a list of **labels** and a matching list of **heights** — `plt.bar(labels,
heights)`. A throwaway example:

In [ ]:
plt.bar(["apples", "pears", "plums"], [3, 5, 2]); plt.ylabel("count"); plt.show()

### 🎯 Task 10 — how (im)balanced is the target?

Now do exactly that for the target. `counts` below already holds the height for each class — give
`plt.bar` two labels and those heights.

In [ ]:
counts = historical_df[TARGET].value_counts().sort_index()
non_failure = counts[0]    # rows that did NOT fail (target == 0)
failure     = counts[1]    # rows that failed        (target == 1)

# 🎯 draw a bar chart: labels ["healthy (0)", "failure (1)"], heights = the two counts (wrap them in a list)
plt.bar(["healthy (0)", "failure (1)"], ???, color=["#55a868", "#c44e52"])
plt.title("Class balance of the target"); plt.ylabel("rows"); plt.show()

print("Failure share:", round(historical_df[TARGET].mean() * 100, 1), "%")

**This is the heart of the trap.** Only ~3% of rows are failures — so let's reason about what
that does to **accuracy** before we trust it.

Accuracy just asks: *of all rows, what fraction did the model label correctly?*

$$\text{accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}$$

Now imagine the laziest possible model: it **always predicts "healthy"**, no matter the sensors.
It gets every healthy row right and every failure row wrong — so its accuracy is simply the
**share of rows that are healthy**.

### 🎯 Predict before you slide
For each class balance below, work out (on paper) what accuracy that "always healthy" model would
score. *Failures rare → healthy share high → accuracy high.*

In [ ]:
pdm_viz.number_quiz("always_healthy")

Now drag the slider to *feel* the same effect continuously: as failures get rarer, the
"always healthy" model's accuracy climbs toward 100%.

But a high accuracy here is hollow, and to say *why* we need two metrics that look **only at the
failures** — the rows we actually care about:
- **Recall** — of all the *real failures*, what fraction did the model catch?
  $\text{recall} = \frac{\text{failures caught}}{\text{all real failures}}$
- **Precision** — of all the rows the model *flagged as failures*, what fraction were real?
  $\text{precision} = \frac{\text{real failures flagged}}{\text{all rows flagged}}$

The "always healthy" model flags **nothing**, so it catches **0** of the failures: **recall = 0%**.
For predictive maintenance that is the metric that matters most — a missed failure means a machine
breaks mid-production, far worse than a false alarm. Accuracy can sit at 97% while recall is 0%.

In [ ]:
pdm_viz.imbalance_slider(historical_df[TARGET].mean() * 100)

### 🎯 Task 11 — correlation heatmap

**Correlation** measures how two columns move together, on a scale from **−1 to +1**:
**+1** = when one goes up the other goes up in lockstep, **−1** = one up means the other down,
**0** = no linear relationship. We look at it to spot **redundant features** (two columns saying
nearly the same thing) and to sanity-check the physics.

**First, a gut-check on what correlation actually means** — decide which statements are true.
A couple of them are *guesses about this data*; we'll compute the real matrix next and see how
those land.

In [ ]:
pdm_viz.true_false_quiz("correlation")

Now compute the correlation matrix of the numeric features with **`.corr()`** and draw it
with the `pdm_viz.heatmap()` helper — and check it against the two guesses above.

In [ ]:
corr = ???               # 🎯 hint: historical_df[NUMERIC].corr()

pdm_viz.heatmap(corr.values, NUMERIC, NUMERIC, "Correlation between sensor features",
                cmap="coolwarm", fmt=".2f")

**Read it carefully — the obvious guess is wrong.** You might expect the two *temperatures*
to be near-identical, but **Air temperature** and **Process temperature** are only **weakly**
correlated here (≈ **0.11**): knowing one tells you little about the other. The genuinely strong
relationship is **Rotational speed ↔ Torque** at ≈ **−0.85** — a near-mirror, straight out of the
physics of mechanical power (more torque, lower speed). *Lesson: read the numbers, don't assume.*

### Historical vs. deployment — are we modeling the same world?

Here's where that second file finally matters. A model is only trustworthy if the data it *meets in
production* looks like the data it *learned from*. The team built everything on `historical` (in
Part 4 we'll formally split it into train/test); but the model will actually run on `deployment_df`,
a batch from a **newer production line**. Before trusting any score, let's compare the two batches.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))

ax1.hist(historical_df["Tool wear [min]"].dropna(), bins=40, alpha=0.6,
         label="historical (training)", color="#6f7bf0", density=True)
ax1.hist(deployment_df["Tool wear [min]"].dropna(), bins=40, alpha=0.6,
         label="deployment (new line)", color="#dd8452", density=True)
ax1.set_title("Tool wear distribution"); ax1.set_xlabel("minutes"); ax1.legend()

mix = pd.DataFrame({
    "historical": historical_df["Type"].map(TYPE_CANON).value_counts(normalize=True),
    "deployment": deployment_df["Type"].map(TYPE_CANON).value_counts(normalize=True),
}).reindex(["L", "M", "H"])
mix.plot(kind="bar", ax=ax2, color=["#6f7bf0", "#dd8452"])
ax2.set_title("Machine-type mix"); ax2.set_ylabel("share"); ax2.set_xlabel("Type")
plt.tight_layout(); plt.show()

print("The new line runs more worn tools and many more 'H' machines than training data had.")
print("=> distribution shift. A model trusted on history may not transfer. We'll test this.")

---
# Part 4 — Prepare the data *honestly*

Now we rebuild the dataset the right way: remove leakage & IDs, drop duplicates, fix categories,
handle missing and impossible values, and scale the features.

### One last move — patching gaps with `.median()` and `.fillna()`

A gap (`NaN`) we can't just leave — most models refuse to train on it. The simplest *honest*
patch is to fill each gap with that column's **typical** value. Two new Series moves do it:
- **`.median()`** *aggregates* a column down to its middle value (more robust to outliers than the mean).
- **`.fillna(value)`** *transforms* a column, swapping every `NaN` for `value` and leaving real cells untouched.

Watch them on a tiny Series:

In [ ]:
toy = pd.Series([10, None, 7, None, 13])
print("median (the NaNs are ignored):", toy.median())
print(toy.fillna(toy.median()))      # every NaN becomes the median; the real values stay put

### Putting the fixes together — `clean_frame`

We've now *found* every problem by hand. `clean_frame` simply **replays those exact fixes** in
order so we can apply them to any batch (we'll reuse it on the deployment data later). Read it —
each step is one move you already met:

In [ ]:
def clean_frame(df, medians=None):
    out = df.copy()

    # (Task 7) collapse the messy spellings into one canonical grade per machine
    out["Type"] = out["Type"].map(TYPE_CANON)

    # (Task 8) blank out the impossible sensor readings so they get patched, not trusted.
    # `df.loc[row_mask, "col"] = value` writes `value` into one column, but only on the rows
    # the boolean mask picks out (here the impossible ones from Task 8) — the rest stay put:
    out.loc[out["Torque [Nm]"] < 0, "Torque [Nm]"] = np.nan
    out.loc[out["Rotational speed [rpm]"] == 0, "Rotational speed [rpm]"] = np.nan
    out.loc[out["Air temperature [K]"] > 500, "Air temperature [K]"] = np.nan

    # (just above) fill every remaining gap with that column's median.
    # `medians` lets us REUSE the training medians on the deployment batch later.
    if medians is None:
        medians = {c: out[c].median() for c in NUMERIC}
    for c in NUMERIC:
        out[c] = out[c].fillna(medians[c])

    return out, medians

# Drop duplicates (Task 6) BEFORE anything else, then clean.
hist_clean, train_medians = clean_frame(historical_df.drop_duplicates())
print("rows after dedup + clean:", len(hist_clean))
print("any NaN left?", bool(hist_clean[NUMERIC].isna().any().any()))

### 🎯 Task 12 — drop the columns that must never be features

Here are all the columns in the cleaned table (so you don't have to scroll up):

In [ ]:
print("All columns in the cleaned table:")
for c in hist_clean.columns:
    print("  -", c)

Most of these are honest sensor features — but a few must **never** become model inputs:
the column that *encodes the outcome* (leakage), the two arbitrary *identifiers*, and the
**target** itself. Drop exactly those with the **`.drop(columns=[...])`** move from Task 2, so
only legitimate features remain. We then one-hot encode `Type` for you.

> 💡 Why one-hot encode `Type` instead of mapping L/M/H → 0/1/2? Because 0/1/2 invents a fake
> *ordering and spacing* the model would take literally. One-hot keeps the grades independent.

In [ ]:
# 🎯 the columns to drop: they leak the answer, are arbitrary IDs, or are the target itself
DROP_COLS = [ ??? ]       # 🎯 the leakage column, the two ID columns, and TARGET

features = hist_clean.drop(columns=DROP_COLS)        # keep only honest features
# `Type` is still text (L/M/H), but models need numbers. One-hot encoding replaces that one
# column with separate 0/1 indicator columns (Type_L, Type_M) — we do this step for you:
# (for the curious: pd.get_dummies makes one 0/1 column per category; drop_first=True omits the
#  first one, since "not L and not M" already implies H — so two columns suffice for three grades)
X = pd.get_dummies(features, columns=["Type"], prefix="Type", drop_first=True)
y = hist_clean[TARGET]

# self-check: none of the dropped columns should have survived into X
for col in DROP_COLS:
    assert col not in X.columns, f"{col} is still in the features!"
print("Feature columns:", list(X.columns))

### One thing the engineers *did* get right: scaling

Look at the raw range of each feature. They are on **wildly different scales** — and that matters
a lot for a model like logistic regression.

In [ ]:
# .describe() again — but we only want the min and max rows, so transpose and pick those two columns
print( X[NUMERIC].describe() ??? )   # 🎯 transpose it (.T), then keep only the "min" and "max" columns

### 🧠 Quick check — why scale?

The engineers' pipeline scaled the features, and that was a good call. Make sure you know *why*
before we copy it.

In [ ]:
pdm_viz.true_false_quiz("scaling")

**Split, then scale.** We fit the scaler on the **training split only** — fitting it on the
test data would let test information leak into training (a subtle but real form of leakage).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

scaler = StandardScaler().fit(X_train)          # FIT on train only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("train:", X_train_s.shape, "· test:", X_test_s.shape)

---
# Part 5 — Train an honest model

Same algorithm the engineers used (logistic regression) — but on **clean, leakage-free** data.

### 🎯 Task 13 — fit and evaluate

Train the model, then predict on the test set. Recall the **`.fit(...)`** method every sklearn
model is trained with, from the earlier sklearn notebook — and make sure you train it on the
**training** split, not the test one. The `classification_report` then shows precision/recall/F1
per class.

In [ ]:
model = LogisticRegression(max_iter=2000, random_state=0)
???                       # 🎯 train the model — recall the .fit() call from the sklearn notebook (on the correct split!)

y_pred = model.predict(X_test_s)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\n", classification_report(y_test, y_pred, target_names=["healthy", "failure"], digits=3))

### What just happened?

| | Engineers' model | Your honest model |
|---|---|---|
| Data | leakage + duplicates | cleaned, leakage removed |
| **Accuracy** | ~1.00 🎉 | ~0.97 |
| **Failures actually caught (recall)** | (fake) | **~0.20** |

The accuracy barely dropped — but look at **recall on the `failure` class**: the honest model
catches only about **1 in 5** real failures. The "always healthy" baseline scores ~0.97 accuracy
too, while catching **none**. **Accuracy was hiding total failure on the job that matters.**

**Can we do better on failures?** One lever: tell the model the classes matter equally with
`class_weight="balanced"`. Watch recall jump — and accuracy *fall*. There is no free lunch.

In [ ]:
balanced = LogisticRegression(max_iter=2000, random_state=0, class_weight="balanced")
balanced.fit(X_train_s, y_train)
bp = balanced.predict(X_test_s)
print("Balanced model accuracy:", round(accuracy_score(y_test, bp), 3))
print(classification_report(y_test, bp, target_names=["healthy", "failure"], digits=3))

---
# Part 6 — Interpret performance like a manager

The right metric depends on **what an error costs the business.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (mdl, name) in zip(axes, [(model, "Honest (default)"), (balanced, "Balanced")]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, mdl.predict(X_test_s), display_labels=["healthy", "failure"],
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout(); plt.show()
print("Top-left/bottom-right = correct. Bottom-left = MISSED failures (the costly ones).")

### 🕹️ Interactive — what does this model cost?

For machine maintenance the two mistakes are **not** equal:
- A **false alarm** → a technician checks a healthy machine. Annoying, modest cost.
- A **missed failure** → the machine breaks mid-production. Downtime, damage, expensive.

Under the hood the model doesn't output a hard 0/1 — it outputs a **confidence**, a probability
between **0 and 1** that the machine will fail. We turn that into a decision with a **threshold**:
flag "failure" whenever the probability is at or above it. The default 0.5 is just a convention —
**we are free to move it**. Lower the threshold and the model flags more rows: **recall goes up**
(we miss fewer failures) but **precision goes down** (more of the flags are false alarms). So you
can dial the threshold to hit a **target recall/precision**, or to **minimise the empirical cost**.

Slide the costs and the **decision threshold** below. Watch the total cost, the recall, and the
precision move together — there's no setting that maxes out all three at once.

In [ ]:
# use the honest model's predicted failure probabilities on the test set
y_score_test = model.predict_proba(X_test_s)[:, 1]
pdm_viz.cost_explorer(y_test.to_numpy(), y_score_test)

**Takeaway:** if a missed failure is far costlier than a false alarm, you should *lower the
threshold* (or use the balanced model) to catch more failures — accepting more false alarms.
The "best" model is the one with the lowest **business cost**, not the highest accuracy.

### The final question: should this go live on the new line?

We tuned and judged everything on **historical** data. But the model will run on the **new
production line** (`deployment_df`) — which we saw has more worn tools and more `H` machines.
Let's evaluate the honest model there, reusing the **training** medians and feature columns.

In [ ]:
dep_clean, _ = clean_frame(deployment_df, medians=train_medians)
X_dep = pd.concat([dep_clean[NUMERIC],
                   pd.get_dummies(dep_clean["Type"], prefix="Type", drop_first=True)],
                  axis=1).reindex(columns=X.columns, fill_value=0)
y_dep = dep_clean[TARGET]
X_dep_s = scaler.transform(X_dep)

rep_hist = classification_report(y_test, model.predict(X_test_s), output_dict=True)
rep_dep = classification_report(y_dep, model.predict(X_dep_s), output_dict=True)
print(f"{'':22}{'accuracy':>10}{'failure recall':>16}{'failure precision':>19}")
for name, r, acc in [("historical test", rep_hist, accuracy_score(y_test, model.predict(X_test_s))),
                     ("NEW deployment line", rep_dep, accuracy_score(y_dep, model.predict(X_dep_s)))]:
    print(f"{name:22}{acc:>10.3f}{r['1']['recall']:>16.3f}{r['1']['precision']:>19.3f}")

**Notice the trap one more time.** Accuracy on the new line looks just as good (~0.97) — yet
the model catches an even *smaller* share of real failures than it did in testing. The metric you'd
quote in a meeting (accuracy) is **blind to the degradation** that actually matters.

**Verdict:** *No — don't ship it Monday.* Not because the model is worthless, but because:
1. it was validated on data that doesn't match the deployment world (**distribution shift**) — in
   the *next* exercise notebook, on information theory, you'll see how to **quantify** exactly how
   far two distributions have drifted apart, using the **KL divergence** — and
2. accuracy hid that it misses most failures — the exact event it exists to prevent.

A responsible next step: retrain on data representative of the new line, choose a threshold from
**business cost**, and **monitor** recall in production.

---
# 🎓 Wrap-up — your "before you trust it" checklist

In [ ]:
pdm_viz.checklist()

### What the engineering team got wrong

1. **Left `Failure Type` in the features** → target leakage → a fake ~100% score.
2. **Trusted accuracy** on a 3%-failure problem, where "always healthy" already scores ~97%.
3. **Didn't check distribution shift** between training data and the new production line.
